# Solicitation Analysis

Load SAM.gov opportunity data, prepare it for analysis, and run the solicitation analyzer.

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)

DATA_DIR = Path("../data")
ATTACHMENTS_DIR = DATA_DIR / "attachments"

## Load opportunities into a DataFrame

In [ ]:
with open(DATA_DIR / "opportunities.json") as f:
    opportunities_raw = json.load(f)

print(f"Loaded {len(opportunities_raw)} opportunities")

In [ ]:
# Flatten the JSON into a DataFrame, keeping nested fields as-is for now
df = pd.json_normalize(opportunities_raw)
df["postedDate"] = pd.to_datetime(df["postedDate"])
df["responseDeadLine"] = pd.to_datetime(df["responseDeadLine"], utc=True, errors="coerce")

print(f"{len(df)} rows, {len(df.columns)} columns")
df[["noticeId", "title", "type", "postedDate", "naicsCode", "office"]].head(10)

In [ ]:
df["type"].value_counts()

## Filter to solicitation types only

Keep only the opportunity types we care about for analysis. Delete attachments on disk that belong to excluded rows.

In [ ]:
KEEP_TYPES = {"Combined Synopsis/Solicitation", "Solicitation", "Presolicitation"}

# Identify rows to drop and collect their attachment filenames for cleanup
df_excluded = df[~df["type"].isin(KEEP_TYPES)]
excluded_files = []
for files in df_excluded["downloadedFiles"]:
    if isinstance(files, list):
        excluded_files.extend(files)

# Delete attachment files belonging to excluded opportunity types
deleted = 0
for filename in excluded_files:
    path = ATTACHMENTS_DIR / filename
    if path.exists():
        path.unlink()
        deleted += 1

print(f"Removed {deleted} attachment files from excluded types")

# Filter DataFrame to only the types we care about
df = df[df["type"].isin(KEEP_TYPES)].copy().reset_index(drop=True)
print(f"{len(df)} opportunities remaining after filtering to: {KEEP_TYPES}")
df["type"].value_counts()

## Resolve attachment file paths

Map each opportunity's `downloadedFiles` list to absolute paths in `data/attachments/` and filter to files that actually exist on disk.

In [ ]:
from spotting_concrete_boats.documents import can_process, compute_attachment_stats


def resolve_attachment_paths(downloaded_files):
    """Return list of existing, processable attachment paths for an opportunity."""
    if not isinstance(downloaded_files, list):
        return []
    paths = []
    for filename in downloaded_files:
        path = ATTACHMENTS_DIR / filename
        if path.exists() and can_process(path):
            paths.append(str(path))
    return paths


df["attachment_paths"] = df["downloadedFiles"].apply(resolve_attachment_paths)
df["attachment_count"] = df["attachment_paths"].apply(len)

# Precompute page counts and file type breakdowns (opens each PDF once)
compute_attachment_stats(df)

print(f"Opportunities with attachments on disk: {(df['attachment_count'] > 0).sum()}")
print(f"Total processable attachments: {df['attachment_count'].sum()}")
print(f"Total PDF pages across all attachments: {df['total_pages'].sum()}")

## Prep for analysis

Filter to opportunities that have either a meaningful description or downloadable attachments — these are the ones worth running through the analyzer.

In [ ]:
from spotting_concrete_boats.documents import build_solicitation_context

# Build structured context for each opportunity from its metadata
df["context"] = df.apply(build_solicitation_context, axis=1)

# Flag opportunities that have substantive content to analyze
df["has_attachments"] = df["attachment_count"] > 0
df["analyzable"] = (df["context"].str.len() > 0) | df["has_attachments"]

print(f"Analyzable opportunities: {df['analyzable'].sum()} / {len(df)}")
print(f"  - With attachments: {df['has_attachments'].sum()}")
print(f"\nSample context:\n{'=' * 80}")
print(df["context"].iloc[0])

In [ ]:
df_to_analyze = df[df["analyzable"]].copy().reset_index(drop=True)
print(f"{len(df_to_analyze)} opportunities ready for analysis")
df_to_analyze[["noticeId", "title", "type", "attachment_count"]].head(10)

## Run the analyzer

Iterate over the filtered opportunities and run the `SolicitationAnalyzer` on each one. Results are collected into a list and merged back into the DataFrame.

In [ ]:
from spotting_concrete_boats.analyzer import (
    SolicitationAnalyzer,
    results_to_dataframe,
    evidence_to_dataframe,
)

analyzer = SolicitationAnalyzer()
print(analyzer)
analyzer.describe_prompts()

In [ ]:
# Run on a single row — change `row_idx` to pick which one
row_idx = 5
row = df_to_analyze.iloc[row_idx]

print(f"[{row_idx + 1}/{len(df_to_analyze)}] {row['title']}")

file_paths = row["attachment_paths"]
context = row["context"] if row["context"] else None

single_result = analyzer.analyze_from_files(file_paths, description=context)
print(single_result)


In [ ]:
results_to_dataframe(single_result)

In [ ]:
all_results = []

for idx, row in df_to_analyze.iterrows():
    print(f"\n[{idx + 1}/{len(df_to_analyze)}] {row['title']}")

    file_paths = row["attachment_paths"]
    context = row["context"] if row["context"] else None

    try:
        results = analyzer.analyze_from_files(file_paths, description=context)
        all_results.append({"noticeId": row["noticeId"], "results": results})
    except Exception as e:
        print(f"  Error: {e}")
        all_results.append({"noticeId": row["noticeId"], "results": None, "error": str(e)})

print(f"\nDone. {sum(1 for r in all_results if r['results'])} / {len(all_results)} succeeded.")

In [ ]:
# View summary for the last successful result
for entry in reversed(all_results):
    if entry["results"]:
        print(f"Result for: {entry['noticeId']}")
        display(results_to_dataframe(entry["results"]))
        break

## View results

In [ ]:
# Build summary and evidence DataFrames for the first successful result as a sanity check
for entry in all_results:
    if entry["results"]:
        print(f"Sample result for: {entry['noticeId']}")
        display(results_to_dataframe(entry["results"]))
        display(evidence_to_dataframe(entry["results"]))
        break

## Load results from batch run

Load per-solicitation result JSON files saved by `scripts/run_analysis.py` or `scripts/run_batch.py`.

In [ ]:
RESULTS_DIR = DATA_DIR / "results"

if RESULTS_DIR.exists():
    saved_results = []
    for result_file in sorted(RESULTS_DIR.glob("*.json")):
        with open(result_file) as f:
            saved_results.append(json.load(f))

    print(f"Loaded {len(saved_results)} saved results from {RESULTS_DIR}")

    # Build combined summary DataFrame
    if saved_results:
        summary_rows = []
        for entry in saved_results:
            if entry.get("results"):
                row_summary = results_to_dataframe(entry["results"])
                row_summary["notice_id"] = entry["notice_id"]
                row_summary["title"] = entry.get("title", "")
                summary_rows.append(row_summary)

        if summary_rows:
            df_summary = pd.concat(summary_rows, ignore_index=True)
            print(f"Combined summary: {len(df_summary)} rows across {len(summary_rows)} solicitations")
            display(df_summary.head(20))
else:
    print(f"No results directory found at {RESULTS_DIR}. Run scripts/run_analysis.py first.")